# agent_framework_devui

## Example 1: a server app with one registered agent

In [ ]:
# Example 1: DevServer builds a FastAPI app; /health answers
from fastapi.testclient import TestClient
from agent_framework import Agent, ChatResponse, ChatResponseUpdate, Content, Message, ResponseStream
from agent_framework_devui import DevServer

class CannedChatClient:
    """Stands in for a model: the same answer, streamed or not (the agent may ask for either)."""
    additional_properties: dict = {}
    def __init__(self, text):
        self.text = text
    def get_response(self, messages, *, stream=False, options=None, **kwargs):
        if stream:
            async def updates():
                yield ChatResponseUpdate(contents=[Content.from_text(self.text)], role="assistant")
            return ResponseStream(updates(), finalizer=ChatResponse.from_updates)
        async def answer():
            return ChatResponse(messages=Message(role="assistant", contents=[self.text]))
        return answer()

agent = Agent(client=CannedChatClient("hello from devui"), name="smoke_agent", description="answers from a canned client")
server = DevServer(ui_enabled=False, auth_enabled=False)
server.register_entities([agent])
http = TestClient(server.get_app(), base_url="http://127.0.0.1:8080")  # DevUI accepts only loopback Host headers
health = http.get("/health")
assert health.status_code == 200, health.text
print("Example 1:", health.json())

## Example 2: the entity list shows the registered agent

In [ ]:
# Example 2: GET /v1/entities lists what the UI can run
entities = http.get("/v1/entities")
assert entities.status_code == 200, entities.text
names = [e.get("name") for e in entities.json().get("entities", [])]
assert "smoke_agent" in names, entities.json()
print("Example 2:", names)

## Example 3: the OpenAI-compatible API schema is served

In [ ]:
# Example 3: the app's OpenAPI document carries the Responses and Conversations routes
schema = http.get("/openapi.json").json()
paths = sorted(schema["paths"])
assert "/v1/responses" in paths and "/v1/conversations" in paths, paths
print("Example 3:", len(paths), "paths, e.g.", paths[:4])

## Example 4: run the agent through the OpenAI-compatible Responses endpoint

In [ ]:
# Example 4: POST /v1/responses with the agent as the "model" runs it and returns its canned answer
entity_id = next(e["id"] for e in http.get("/v1/entities").json()["entities"] if e.get("name") == "smoke_agent")
reply = http.post("/v1/responses", json={"model": entity_id, "input": "hi", "metadata": {"entity_id": entity_id}, "stream": False})
print("Example 4: status", reply.status_code, "answer contains the canned text:", "hello from devui" in reply.text)
assert reply.status_code == 200 and "hello from devui" in reply.text, reply.text[:400]

## Example 5: `agent_framework.devui` re-exports this package

In [ ]:
# Example 5: the umbrella namespace hands back the same server class
import importlib.metadata
import agent_framework.devui as via_core
import agent_framework_devui as direct

assert via_core.DevServer is direct.DevServer and via_core.serve is direct.serve
print("Example 5:", importlib.metadata.version("agent-framework-devui"), "re-exported through agent_framework.devui")